# IncidentCommander RL — Kaggle shard 2 / 3

**Workload:** every 3rd task starting at index 1
(~127 of 381 scenarios). Trains a LoRA on `Qwen/Qwen2.5-1.5B-Instruct` using a
local `Qwen/Qwen2.5-7B-Instruct` critic. **Both models download from Hugging Face Hub
at notebook startup — no Kaggle Models attachment needed.**

**Required notebook settings** (right-hand sidebar):
- Accelerator: `GPU T4 x2` or `GPU P100`
- Persistence: `Files only` (so `/kaggle/working/` survives restarts)
- Internet: `On` (needed to download the models + clone the repo)

**Optional** (only if you want intermediate checkpoint upload to your HF
repo): Add-ons → Secrets → add `HF_TOKEN` and toggle it on.

**Output:** `/kaggle/working/adapter_kaggle2.zip` — download from
the sidebar after the run finishes. Combine all 3 with
`scripts/merge_lora_adapters.py` on your laptop.


## 1. GPU + path sanity

In [ ]:
import subprocess
print('--- GPU ---')
subprocess.run(['nvidia-smi', '-L'], check=False)
import torch
print('CUDA OK?', torch.cuda.is_available(), '| device:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 2. Install deps (Kaggle has torch/transformers preinstalled — we just pin compatible versions)

In [ ]:
%pip install -q \
    "transformers==4.46.3" \
    "peft==0.13.2" \
    "accelerate==1.1.1" \
    "bitsandbytes==0.45.5" \
    "trl==0.12.1" \
    "huggingface_hub>=0.25,<1.0" \
    "pydantic>=2,<3" \
    "datasets" "sentencepiece" "protobuf" "safetensors"

## 3. Download actor + critic from Hugging Face Hub

In [ ]:
import os, pathlib
from huggingface_hub import snapshot_download

# Optional HF_TOKEN — only needed if you have it set in Kaggle Secrets.
# Qwen 2.5 instruct models are public, so anonymous download works fine.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN attached from Kaggle Secrets')
except Exception:
    print('No HF_TOKEN — anonymous download (Qwen 2.5 instruct is public).')

CACHE = '/kaggle/working/hf-cache'
pathlib.Path(CACHE).mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME']            = CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = CACHE

print('downloading actor : Qwen/Qwen2.5-1.5B-Instruct  (~3 GB) ...')
ACTOR_PATH = snapshot_download(
    repo_id='Qwen/Qwen2.5-1.5B-Instruct',
    cache_dir=CACHE,
    allow_patterns=['*.json', '*.safetensors', '*.txt', 'tokenizer*'])

print('downloading critic: Qwen/Qwen2.5-7B-Instruct  (~15 GB) ...')
CRITIC_PATH = snapshot_download(
    repo_id='Qwen/Qwen2.5-7B-Instruct',
    cache_dir=CACHE,
    allow_patterns=['*.json', '*.safetensors', '*.txt', 'tokenizer*'])

print('actor :', ACTOR_PATH)
print('critic:', CRITIC_PATH)

## 4. Clone the repo (public GitHub)

In [ ]:
import os, subprocess, pathlib
WORK = '/kaggle/working/incident-commander'
if not pathlib.Path(WORK).exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/r1cksync/meta-rl-hack.git', WORK], check=True)
os.chdir(WORK)
print('cwd =', os.getcwd())

## 5. Configure run (shard, paths, env vars)

In [ ]:
import os

os.environ['INCIDENT_COMMANDER_MOCK'] = 'true'
os.environ['IC_ACTOR_MODEL']     = ACTOR_PATH
os.environ['IC_CRITIC_PROVIDER'] = 'local'        # 7B critic on the same GPU
os.environ['IC_CRITIC_MODEL']    = CRITIC_PATH
os.environ['IC_TASK_MODE']       = 'all'           # full 381 corpus
os.environ['IC_TASK_SHARDS']     = '3'
os.environ['IC_TASK_SHARD']      = '1'
os.environ['IC_TOTAL_UPDATES']   = '60'            # ~6h on T4 / P100
os.environ['IC_ROLLOUTS']        = '3'
os.environ['IC_MAX_STEPS']       = '12'
os.environ['IC_CKPT_EVERY']      = '15'
os.environ['IC_RUN_NAME']        = 'kaggle2'

print('actor :', os.environ['IC_ACTOR_MODEL'])
print('critic:', os.environ['IC_CRITIC_MODEL'])
print('shard :', os.environ['IC_TASK_SHARD'], '/', os.environ['IC_TASK_SHARDS'])

## 6. Train

In [ ]:
# The training script runs to completion. tqdm progress + ETA are streamed
# to stdout. Kaggle truncates very long outputs — adapter checkpoints are
# always written to /kaggle/working/incident-commander/colab/logs/ regardless.
import subprocess, sys
result = subprocess.run([sys.executable, 'scripts/run_training.py'],
                         check=False)
print('exit code:', result.returncode)

## 7. Package outputs for download

In [ ]:
import shutil, glob, pathlib

LOGS = pathlib.Path('colab/logs')
finals = sorted(LOGS.glob('adapter_kaggle2_final'))
ckpts  = sorted(LOGS.glob('adapter_kaggle2_u*'))
keep   = (finals or ckpts)
assert keep, 'No adapter directories found — check the training cell output for errors.'
src = keep[-1]
print('packaging', src)

dst = pathlib.Path('/kaggle/working/adapter_kaggle2.zip')
shutil.make_archive(str(dst.with_suffix('')), 'zip', root_dir=src)
print('zipped to', dst, 'size:', dst.stat().st_size, 'bytes')

# Also copy the JSON training log for plotting on your laptop.
for j in glob.glob('colab/logs/training_kaggle2*.json'):
    shutil.copy(j, '/kaggle/working/')
print('files in /kaggle/working/:')
for f in sorted(pathlib.Path('/kaggle/working/').iterdir()):
    if f.name == 'hf-cache': continue   # don't list the model cache
    print(' ', f.name, f.stat().st_size if f.is_file() else '<dir>')

## Done

Download `adapter_kaggle2.zip` from the **Output** tab on the
right.  Repeat for the other two shards (notebooks 2 and 3), then on your
laptop run:

```powershell
python scripts/merge_lora_adapters.py `
    --inputs ./adapter_kaggle1 ./adapter_kaggle2 ./adapter_kaggle3 `
    --output ./adapter_merged
```

The merged adapter loads with the standard `peft` API on top of
`Qwen/Qwen2.5-1.5B-Instruct`.
